In [ ]:
!pip install numpy pandas torch scikit-learn tensorflow deepctr-torch
#################################
# Import Libraries
#################################
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import ParameterGrid
from tensorflow.keras.preprocessing.sequence import pad_sequences
from deepctr_torch.inputs import SparseFeat, get_feature_names
from deepctr_torch.models import DeepFM
import matplotlib.pyplot as plt


#################################
# Utility Functions
#################################
def compute_mahalanobis_distance(mean, cov_inv, embedding):
    diff = embedding - mean
    return np.sqrt(diff.T @ cov_inv @ diff)


class DriftDetector:
    def __init__(self, embedding_dim, alpha=0.01):
        self.alpha = alpha
        self.mean = np.zeros(embedding_dim)
        self.cov = np.eye(embedding_dim)
        self.count = 0

    def update(self, embedding):
        if self.count == 0:
            self.mean = embedding
            self.cov = np.eye(len(embedding))
        else:
            self.mean = (1 - self.alpha) * self.mean + self.alpha * embedding
            diff = embedding - self.mean
            self.cov = (1 - self.alpha) * self.cov + self.alpha * np.outer(
                diff, diff)
        self.count += 1

    def detect_drift(self, embedding):
        return compute_mahalanobis_distance(self.mean,
                                            np.linalg.pinv(self.cov),
                                            embedding)


#################################
# Load and Preprocess Data
#################################
data = pd.read_json("experiment_1/Amazon_Fashion.jsonl", lines=True)
data = data[data["rating"] != 3]
data = data[data["timestamp"] > '2015-01-01']
data["year"] = data["timestamp"].apply(lambda x: x.year)
data["month"] = data["timestamp"].apply(lambda x: x.month)
data["day"] = data["timestamp"].apply(lambda x: x.day)
data["rating"] = data["rating"].transform(lambda x: 1 if x > 3 else 0)

data.set_index(data["timestamp"], inplace=True)
data.sort_index(inplace=True)

#################################
# Define Features and Model
#################################
sparse_features = ["parent_asin", "user_id", "year", "month", "day"]
target = ['rating']

# Label Encoding for sparse features
for feat in sparse_features:
    lbe = LabelEncoder()
    data[feat] = lbe.fit_transform(data[feat])

# Define feature columns
embedding_dim = 8
fixlen_feature_columns = [
    SparseFeat(feat, data[feat].nunique(), embedding_dim=embedding_dim)
    for feat in sparse_features
]
linear_feature_columns = fixlen_feature_columns
dnn_feature_columns = fixlen_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

# Prepare model input
model_input = {name: data[name] for name in sparse_features}

# Define and compile the model
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary',
               device=device)
model.compile("adam", metrics=["AUC"], loss='binary_crossentropy')

# Train the model
history = model.fit(
    model_input,
    data[target].values,
    batch_size=1024,
    epochs=6,
    verbose=2,
    validation_split=0.1
)

#################################
# Hyperparameter Tuning
#################################
# Define parameter grid
param_grid = {
    "alpha": [0.01, 0.05, 0.1],
    "window_size": [5, 10, 20],
    "threshold_multiplier": [1.5, 2, 3]
}

# Define ground truth
true_drifts = np.zeros(300)
true_drifts[200:] = 1

# Initialize variables for results
best_params = None
best_fpr = float('inf')
best_latency = float('inf')

# Grid search
for params in ParameterGrid(param_grid):
    print(f"Testing params: {params}")
    alpha = params["alpha"]
    window_size = params["window_size"]
    threshold_multiplier = params["threshold_multiplier"]

    tracker = DriftDetector(embedding_dim, alpha=alpha)
    drift_distances = []
    thresholds = []
    false_positive_rate = []
    latencies = []

    with torch.no_grad():
        for batch_index, batch in enumerate(
                np.array_split(data[sparse_features], 300)):
            batch_input = {key: batch[key].values for key in sparse_features}
            embeddings = model.predict(batch_input)
            mean_embedding = np.mean(embeddings, axis=0)

            tracker.update(mean_embedding)
            drift_distance = tracker.detect_drift(mean_embedding)
            drift_distances.append(drift_distance)

            if len(drift_distances) > window_size:
                recent_dists = drift_distances[-window_size:]
                mean_dist = np.mean(recent_dists)
                std_dist = np.std(recent_dists)
                threshold = mean_dist + threshold_multiplier * std_dist
            else:
                threshold = np.mean(
                    drift_distances) + threshold_multiplier * np.std(
                    drift_distances)

            thresholds.append(threshold)
            is_drift_detected = drift_distance > threshold

            # Calculate FPR and latency
            actual_drift = true_drifts[batch_index]
            if batch_index < 200:
                false_positive_rate.append(is_drift_detected)
            else:
                false_positive_rate.append(0)
                if actual_drift == 1 and is_drift_detected:
                    latencies.append(batch_index - 200)

    # Evaluate results
    fpr = np.mean(false_positive_rate)
    avg_latency = np.mean(latencies) if latencies else float('inf')
    print(f"Params: {params}, FPR: {fpr:.4f}, Latency: {avg_latency:.4f}")

    if fpr < best_fpr or (fpr == best_fpr and avg_latency < best_latency):
        best_params = params
        best_fpr = fpr
        best_latency = avg_latency

print(
    f"Best Params: {best_params}, Best FPR: {best_fpr:.4f}, Best Latency: {best_latency:.4f}")

#################################
# Visualization for Best Params
#################################
# Visualize with best parameters
alpha = best_params["alpha"]
window_size = best_params["window_size"]
threshold_multiplier = best_params["threshold_multiplier"]

plt.figure(figsize=(12, 6))
plt.plot(drift_distances, label="Distances")
plt.plot(thresholds, label="Thresholds", linestyle="--")
plt.scatter(np.arange(len(drift_distances))[
                np.array(drift_distances) > np.array(thresholds)],
            np.array(drift_distances)[
                np.array(drift_distances) > np.array(thresholds)],
            color='red', s=10, label="Drift Points")
plt.xlabel("Batch Index")
plt.ylabel("Mahalanobis Distance")
plt.title(f"Drift Detection with Best Params: {best_params}")
plt.legend()
plt.savefig("best_drift_detection.png")
plt.show()


mps
Train on 1950237 samples, validate on 216694 samples, 1905 steps per epoch
